# Tutorial export metadata on WP6 database
> In this notebook, we will use: 
> * data dictionary excel: clinical reference
> * CSV export file: mock up patient data structure
> * SAS file: official export column labels and informats

In [40]:
import importlib
from pathlib import Path
import utils_database

importlib.reload(utils_database)

from utils_database import *

## Read the data dictionary
> extract one row per variable using `Référence` as the variable name
> 
> we read here the data dictionary file `24ACE-FH-EARLY_DHM_v105_20260424.xls`, that provide: 
> - `Id`: internal variable identifier
> - `Référence`: variable name used in exports
> - `Saisie`: entry requirement/status
> - `Type`: variable type
> - `Format`: coding or value specification
> - `Intitulé`: human-readable variable description

In [41]:
df = pd.read_excel("24ACE-FH-EARLY_DHM_v105_20260424.xls", sheet_name="Data Handling Manual FH-EARLY")
# print(df.head())
print(len(df))
variables = df[df["Référence"].notna()].copy()

variables = variables[["Id", "Référence", "Saisie", "Type", "Format", "Intitulé"]]

237


## Read the (mock) export `.csv` files
> CSV files are ANSI/Latin-1 encoded and semicolon-separated: they represent the expected structure of the future real import. 
> The export is divided into multiple CSV tables corresponding to different study modules:
> - `INC`
> - `ML`
> - `TAB_BS`
> - `TAB_FH`

In [42]:
export_folder = Path("export-example")

inc = read_export_csv(export_folder / "INC_ANSI.csv")
ml = read_export_csv(export_folder / "ML_ANSI.csv")
tab_bs = read_export_csv(export_folder / "TAB_BS_ANSI.csv")
tab_fh = read_export_csv(export_folder / "TAB_FH_ANSI.csv")

export_tables = {"INC": inc, "ML": ml, "TAB_BS": tab_bs, "TAB_FH": tab_fh}

## Map dictionary variables into export columns
> Question: For each variable in the dictionary, what columns appear in the export CSV? 
> For instance (examples mentioned by Alain in the email): 
> * `BRTHDAT` in dictionary became `BRTHDAT_D, BRTHDAT_M, BRTHDAT_Y, BRTHDAT` in .csv 
> * numeric in dictionary may include an additional `_V` column in .csv 
> * checkbox variables may expand into multiple `_C1, _C2...` in .csv 
> 
> This is dictionary-centric mapping (as many row as the ones in the dictionary)
### Checkbox variables
> `INC_ANSI.csv` do not contain checkbox examples, `ML_ANSI.csv` does
> A checkbox variable allow selecting multiple answer simultaneosly. Something like: Which sympthoms? 
> * [] fever
> * [] cough
> * [] headache
>
> In the email they said that variable marked as checkbox in the dictionary become: `Variable_C1`, `Variable_C2`, `Variable_C3`in the `.csv`
### Mismatched variables
> Some variables in the data dictionary do not appear in the exported `.csv` file. For instance: 
> * type `Label` are never in the .csv (which are probably not patient data? need to check) 


In [43]:
# example if you want to find the export columns for a SPECIFIC given variable name
for var in ["SEX", "BRTHDAT", "AGE"]:
    print(var, "->", find_export_columns(var, inc.columns))

# ml columns
print("FH7ALS ->", find_export_columns("FH7ALS", ml.columns))

SEX -> ['SEX']
BRTHDAT -> ['BRTHDAT_D', 'BRTHDAT_M', 'BRTHDAT_Y', 'BRTHDAT']
AGE -> ['AGE', 'AGE_V']
FH7ALS -> ['FH7ALS_C1', 'FH7ALS_C2', 'FH7ALS_C3', 'FH7ALS_C4', 'FH7ALS_C5', 'FH7ALS_C6', 'FH7ALS_C7', 'FH7ALS_C8', 'FH7ALS_C9', 'FH7ALS_C10', 'FH7ALS_C11', 'FH7ALS_C12', 'FH7ALS_C13', 'FH7ALS_C14']


In [44]:
# modify export_table depending on what you want to check
export_tables = {"INC": inc, "ML": ml, "TAB_BS": tab_bs, "TAB_FH": tab_fh,}

# mapped dictionary
mapped = create_dictionary_export_map(variables, export_tables)

# miamatched columns
summary, mismatched = summarize_dictionary_export_map(mapped)

### Add encoding for variable type that is in english and useful for ML
> classify_variable_type` converts the original dictionary variable types into simpler categories:
> - `Date`: `date`
> - `Entier`: `numeric`
> - `Réel`:  `numeric`
> - `Radio buttons`: `categorical`
> - `Combo box`: `categorical`
> - `Cases à cocher`: `checkbox`
> - `Texte`: `text`
> - `Label`: `label`
> - `Tableau`: `table`
> - missing/unknown values: `unknown`

In [45]:
mapped["Type_english"] = mapped["Type"].apply(classify_variable_type)
mapped[["Référence", "Type", "Type_english", "Intitulé", "INC_export_cols", "ML_export_cols", "TAB_BS_export_cols", "TAB_FH_export_cols"]].sample(10)

,Référence,Type,Type_english,Intitulé,INC_export_cols,ML_export_cols,TAB_BS_export_cols,TAB_FH_export_cols
215,TTT2YN,Radio buttons,categorical,Statin,[],[TTT2YN],[],[]
71,CD5NUM,Réel,numeric,BMI,[],"[CD5NUM, CD5NUM_V]",[],[]
12,INF1YN,Radio buttons,categorical,Patient deceased at the time of inclusion,[INF1YN],[],[],[]
205,BS38YN,Radio buttons,categorical,Cardiac Ultrasound*,[],[BS38YN],[],[]
137,BS4NUM,Réel,numeric,TAG1,[],"[BS4NUM, BS4NUM_V]",[],[]
195,BS35A1TXT,Texte,text,If yes: cell type*,[],[BS35A1TXT],[],[]
191,BS34A1TXT,Texte,text,"If Other, describe",[],[BS34A1TXT],[],[]
150,BS8ADT,Date,date,Date,[],"[BS8ADT_D, BS8ADT_M, BS8ADT_Y, BS8ADT]",[],[]
20,INF1B5NUM,Réel,numeric,Version of NINO sent/delivered,"[INF1B5NUM, INF1B5NUM_V]",[],[],[]
32,CB5YN,Radio buttons,categorical,Agreement for both entry into the FH patient r...,[CB5YN],[],[],[]


### Add encoding for categorical variable with the meaning of the cateogory
> The function `parse_code_labels` extracts categorical value mappings from the dictionary `Format` column.
> 
> For `categorical` variables: `code_labels` list the possible categories/labels
> Example:
> `1 = Male`, `2 = Female`
> 
> becomes:
> `{1: "Male", 2: "Female"}`
> 
> For `checkbox` variables: `code_labels` lists all possible checkbox options
> 
> The export creates mulitple `_C1`, `_C2`... columns and each column is binary
> Example:
> For `FH7ALS` we have `1 = Mother`, `2 = Father`, `3 = Child`
> 
> becomes:
> `{1: "Mother", 2: "Father", 3: "Child"}`
> that export `FH7ALS_C1`, `FH7ALS_C2` and `FH7ALS_C3`
> 
> For `checkbox` variables: `code_labels` lists all possible checkbox options


In [46]:
mapped["code_labels"] = mapped["Format"].apply(parse_code_labels)
mapped[["Référence", "Type", "Type_english", "Format", "code_labels"]].sample(20)

,Référence,Type,Type_english,Format,code_labels
143,BS7NUM,Réel,numeric,"Format : ####,N\nMin = 0.0\nMax = 9999.0",{}
49,CNIFR1YN,Radio buttons,categorical,0 = No \n1 = Yes,"{0: 'No', 1: 'Yes'}"
82,MH23NUM,Entier,numeric,Taille : 2\nMax = 5,{}
221,TTT4LS,Cases à cocher,checkbox,1 = antiplatelet \n2 = nitrates \n3 = anticoag...,"{1: 'antiplatelet', 2: 'nitrates', 3: 'anticoa..."
146,BS8UNIT,Combo box,categorical,1 = mg/dL \n2 = g/L,"{1: 'mg/dL', 2: 'g/L'}"
110,MH16LYN,Radio buttons,categorical,1 = Yes \n0 = No,"{1: 'Yes', 0: 'No'}"
217,TTT2BLS,Radio buttons,categorical,4 = 4 \n5 = 5 \n10 = 10 \n20 = 20 \n40 = 40 \n...,"{4: '4', 5: '5', 10: '10', 20: '20', 40: '40',..."
93,MH5CYN,Radio buttons,categorical,1 = Yes \n0 = No,"{1: 'Yes', 0: 'No'}"
182,TABBS1LS,Combo box,categorical,1 = LDLR NM_000527.5 \n2 = APOB NM_000384.3 \n...,"{1: 'LDLR NM_000527.5', 2: 'APOB NM_000384.3',..."
39,CSTTR1YN,Radio buttons,categorical,1 = Yes \n0 = No,"{1: 'Yes', 0: 'No'}"


In [47]:
# SANITY check that code labels are captured for all categorical and checkbox variables
mapped[(mapped["Type_english"].isin(["categorical", "checkbox"])) & (mapped["code_labels"].apply(len) == 0)][["Référence", "Type", "Format", "Intitulé"]]

,Référence,Type,Format,Intitulé


In [48]:
# If perhaps you want to save...
# mapped.to_csv("dictionary_export_map.csv", index=False)

In [49]:
# Final output
mapped[["Référence", "Type", "Type_english", "Intitulé", "code_labels", "has_export_match", "n_export_matches", "all_export_matches"]].head(20)

,Référence,Type,Type_english,Intitulé,code_labels,has_export_match,n_export_matches,all_export_matches
2,PROJECTNAME,Label,label,FH-Early,{},False,0,[]
3,SEX,Radio buttons,categorical,Gender,"{1: 'Male', 2: 'Female'}",True,1,[SEX]
4,BRTHDAT,Date,date,Date of birth,{},True,4,"[BRTHDAT_D, BRTHDAT_M, BRTHDAT_Y, BRTHDAT]"
5,AGE,Entier,numeric,Age,{},True,2,"[AGE, AGE_V]"
6,INVNAME,Texte,text,Investigator,{},True,1,[INVNAME]
7,SITEID,Texte,text,Centre,{},True,1,[SITEID]
8,SUBJECT_REF,Texte,text,Patient ID,{},True,5,"[SUBJECT_REF, SUBJECT_REF_CL1, SUBJECT_REF, SU..."
9,RFICDAT,Date,date,Date of inclusion,{},True,4,"[RFICDAT_D, RFICDAT_M, RFICDAT_Y, RFICDAT]"
12,INF1YN,Radio buttons,categorical,Patient deceased at the time of inclusion,"{1: 'Yes', 0: 'No'}",True,1,[INF1YN]
13,INF1AYN,Radio buttons,categorical,"If so, did you verify that there was no object...","{1: 'Yes', 0: 'No'}",True,1,[INF1AYN]


## Parse SAS metadata
> The SAS files associated with the mock export contain technical metadata for the exported CSV columns
>
> This metadata includes:
> - `sas_label`: official human-readable description of each exported column
> - `sas_informat`: SAS typing/formatting information used to read the exported values
> 
> Examples of `sas_informat` values:
> - `$4.` short text/categorical field
> - `$19.` longer text field
> - `best32.` numeric variable
> - `DDMMYY10.` date variable
> 
> The SAS metadata complements the clinical metadata provided by the data dictionary and helps describe the exported database structure more precisely

In [50]:
# parse_all_sas_labels reads all .sas files in the export folder and extracts the official column labels for each export table
# format: dictionary with table names as keys and dictionaries of {column_name: column_label} as values
sas_labels = parse_all_sas_labels(export_folder)
sas_labels.keys()

dict_keys(['ML', 'TAB_FH', 'TAB_BS', 'INC'])

### Create export column catalog
> This step combines CSV column names with metadata extracted from the SAS files
> 
> The resulting `column_catalog` contains one row per exported CSV column, including: table name, column name, `sas_label`, `sas_informat`

In [51]:
sas_informats = parse_all_sas_informats(export_folder)

column_catalog = create_export_column_catalog(
    export_tables,
    sas_labels,
    sas_informats
)

column_catalog.head(-30)

,table,column,sas_label,sas_informat
0,INC,STUDY_ID,Identification étude,$8.
1,INC,COUNTRY_ID,Identifiant pays,$4.
2,INC,EXTRACTION_DATE,Date extraction,$19.
3,INC,SITE_ID,Centre,$4.
4,INC,SUBJECT_ID,Identifiant unique,$4.
...,...,...,...,...
346,TAB_BS,COUNTRY_ID,Identifiant pays,$4.
347,TAB_BS,EXTRACTION_DATE,Date extraction,$19.
348,TAB_BS,SITE_ID,Centre,$4.
349,TAB_BS,SUBJECT_ID,Identifiant unique,$4.


In [52]:
# sanity check to control that the sas catalog is stable
from IPython.display import display

print("1. Total CSV columns vs catalog rows:")
print(sum(len(t.columns) for t in export_tables.values()), len(column_catalog))

print("\n2. Missing SAS labels:")
display(column_catalog[column_catalog["sas_label"].isna()])

print("\n3. Missing SAS informats:")
display(column_catalog[column_catalog["sas_informat"].isna()])

# print("\n4. SAS informat types:")
# display(column_catalog["sas_informat"].value_counts())

print("\n5. Manual sample:")
display(column_catalog.sample(20, random_state=1))

1. Total CSV columns vs catalog rows:
381 381

2. Missing SAS labels:


,table,column,sas_label,sas_informat



3. Missing SAS informats:


,table,column,sas_label,sas_informat



5. Manual sample:


,table,column,sas_label,sas_informat
244,ML,BS21NUM_V,HbA1c*/ Valeur entière ou réelle,best32.
180,ML,BS1A1TXT,"If etc., specify",$21.
277,ML,BS34YN,Gene score calculation,$4.
185,ML,BS4NUM_V,TAG1/ Valeur entière ou réelle,best32.
233,ML,BS39LS_C5,Treatment at the time of analysis / Lomitapide,$4.
314,ML,BS38DNUM_V,CIMT (Carotid Intima Media Thickness) Left (mm...,best32.
138,ML,MH16J1NUM,Carotid endarterectomy age*/ Valeur au format ...,$4.
201,ML,BS10LS,Lp(a)1 unit,$4.
117,ML,MH5B1NUM_V,PTCA Age/ Valeur entière ou réelle,best32.
301,ML,BS37CNUM,Computed Coronary Tomography Angiography (arte...,$4.


## Create master column catalog
> mergeing the export column catalog with the data dictionary
> The resulting `master_catalog` contains one row per actual exported CSV column and combines:
> - CSV table and column name
> - SAS label and informat
> - dictionary reference
> - original dictionary type
> - simplified (english) type
> - human-readable variable description
> - and categorical/checkbox `code_labels`

In [53]:
master_catalog = create_master_column_catalog(column_catalog, variables)

In [54]:
master_catalog["Type_english"] = master_catalog["Type"].apply(classify_variable_type)
master_catalog["code_labels"] = master_catalog["Format"].apply(parse_code_labels)

In [55]:
master_catalog.sample(20, random_state=1)

,table,column,sas_label,sas_informat,Référence,Id,Saisie,Type,Format,Intitulé,Type_english,code_labels
244,ML,BS21NUM_V,HbA1c*/ Valeur entière ou réelle,best32.,BS21NUM,ID102S2V117,O,Réel,"Format : #,#\nMin = 4.0\nMax = 9.9\nUnité = %",HbA1c*,numeric,{}
180,ML,BS1A1TXT,"If etc., specify",$21.,BS1A1TXT,ID102S2V96,O,Texte,Taille : N,"If etc., specify",text,{}
277,ML,BS34YN,Gene score calculation,$4.,BS34YN,ID102S2V135,O,Radio buttons,1 = Yes \n0 = No,Gene score calculation,categorical,"{1: 'Yes', 0: 'No'}"
185,ML,BS4NUM_V,TAG1/ Valeur entière ou réelle,best32.,BS4NUM,ID102S2V98,O,Réel,"Format : ##,##\nMin = 0.0\nMax = 99.0",TAG1,numeric,{}
233,ML,BS39LS_C5,Treatment at the time of analysis / Lomitapide,$4.,BS39LS,ID102S2V231,O,Cases à cocher,00 = Statin \n1 = ezetimibe \n2 = cholestyrami...,Treatment at the time of analysis,checkbox,"{0: 'Statin', 1: 'ezetimibe', 2: 'cholestyrami..."
314,ML,BS38DNUM_V,CIMT (Carotid Intima Media Thickness) Left (mm...,best32.,BS38DNUM,ID102S2V155,O,Réel,"Format : ###,###\nUnité = mm",CIMT (Carotid Intima Media Thickness) Left (mm)*,numeric,{}
138,ML,MH16J1NUM,Carotid endarterectomy age*/ Valeur au format ...,$4.,MH16J1NUM,ID102S2V74,O,Entier,Taille : 2\nMax = 120,Carotid endarterectomy age*,numeric,{}
201,ML,BS10LS,Lp(a)1 unit,$4.,BS10LS,ID102S2V106,O,Combo box,1 = nmol/L \n2 = mg/dL,Lp(a)1 unit,categorical,"{1: 'nmol/L', 2: 'mg/dL'}"
117,ML,MH5B1NUM_V,PTCA Age/ Valeur entière ou réelle,best32.,MH5B1NUM,ID102S2V58,O,Entier,Taille : 2\nMax = 120,PTCA Age,numeric,{}
301,ML,BS37CNUM,Computed Coronary Tomography Angiography (arte...,$4.,BS37CNUM,ID102S2V149,O,Réel,"Format : ###,N\nMin = 0.0\nMax = 100.0\nUnité = %",Computed Coronary Tomography Angiography (arte...,numeric,{}
